# Example: Image serving for agentic retrieval (preview) using Python

This end-to-end notebook creates a blob knowledge source that uses managed ingestion with Azure Content Understanding in Foundry Tools to create semantic chunks, preserve tables, describe document-embedded figures, and extract images. Azure AI Search stores extracted images in an asset store, stores `image_path` references in the generated index, and supplies image content associated with matching results to the multimodal model during answer synthesis.

The retrieve response reports aggregate image-serving statistics, but it doesn't guarantee an extracted-image `image_path` or image bytes. Separately from retrieval, the notebook runs an ordinary wildcard search against the generated index to select an indexed `image_path`, and then downloads that asset to validate application access. The selected path isn't demonstrably associated with a chunk that contributed to the retrieve response.

This sample doesn't use an explicit `OcrSkill` or `normalized_images`. Those elements belong to the classic OCR enrichment pattern.

**Prerequisites:**

- A Python virtual environment created from `requirements.txt` and selected as the notebook kernel.
- For required resources and permissions, see [Surface document-embedded images in agentic retrieval (preview)](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-image-serving#prerequisites).

**Flow:**

1. Create an `azureBlob` knowledge source with `content_extraction_mode="standard"` and an asset store.
1. Poll the generated indexer until managed ingestion succeeds.
1. Independently query the generated index for a nonempty `image_path`.
1. Create a knowledge base with image serving enabled.
1. Retrieve with image serving disabled and enabled, and inspect `ImageServingStatistics`.
1. Download the independently selected indexed image asset with `DefaultAzureCredential`.
1. Delete the knowledge base and knowledge source when you're finished.

**SDK:** `azure-search-documents==12.1.0b2` (REST API `2026-08-01-preview`)

Save `sample.env` as `.env` and fill in the nonsecret resource values before running this notebook.


## Load connections

Before you run this cell, save `sample.env` as `.env` and fill in the values.
You should also create a virtual environment with `requirements.txt` as the dependency list
and select it as the notebook kernel.


In [ ]:
import os
import time

from azure.identity import DefaultAzureCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from dotenv import load_dotenv

load_dotenv(override=True)

endpoint = os.environ["AZURE_SEARCH_ENDPOINT"]
ks_name = os.getenv("AZURE_SEARCH_KNOWLEDGE_SOURCE_NAME", "image-serving-ks")
kb_name = os.getenv("AZURE_SEARCH_KNOWLEDGE_BASE_NAME", "image-serving-kb")
storage_resource_id = os.environ["AZURE_STORAGE_RESOURCE_ID"]
source_container = os.getenv("AZURE_BLOB_SOURCE_CONTAINER", "source-documents")
asset_container = os.getenv("AZURE_BLOB_ASSET_CONTAINER", "image-assets")
foundry_endpoint = os.environ["AZURE_FOUNDRY_ENDPOINT"]
foundry_embedding_deployment = os.environ["AZURE_FOUNDRY_EMBEDDING_DEPLOYMENT"]
foundry_embedding_model = os.environ["AZURE_FOUNDRY_EMBEDDING_MODEL"]
foundry_chat_deployment = os.environ["AZURE_FOUNDRY_CHAT_DEPLOYMENT"]
foundry_chat_model = os.environ["AZURE_FOUNDRY_CHAT_MODEL"]

managed_identity_connection = f"ResourceId={storage_resource_id}"
credential = DefaultAzureCredential()
index_client = SearchIndexClient(endpoint=endpoint, credential=credential)
kb_retrieval_client = KnowledgeBaseRetrievalClient(
    endpoint=endpoint,
    knowledge_base_name=kb_name,
    credential=credential,
)

print(f"Knowledge source: {ks_name}")
print(f"Knowledge base: {kb_name}")

## Create an Azure Blob knowledge source with managed image extraction

The knowledge source starts managed ingestion that:

1. Reads PDF and image files from the source blob container.
1. Extracts text and images without an explicit `OcrSkill` or `normalized_images` pipeline.
1. Stores extracted images in the asset-store blob container.
1. Writes chunks, vectors, and `image_path` references to a generated index.
1. Optionally verbalizes images with the chat model during ingestion.

The `ResourceId=<storage-resource-id>` connection format tells Azure AI Search to use its managed identity. Don't put account keys or connection-string secrets in the notebook.


In [ ]:
from azure.search.documents.indexes.models import (
    AzureBlobKnowledgeSource,
    AzureBlobKnowledgeSourceParameters,
    AzureOpenAIVectorizerParameters,
    KnowledgeBaseAzureOpenAIModel,
)
from azure.search.documents.knowledgebases.models import (
    AIServices,
    AssetStore,
    KnowledgeSourceAzureOpenAIVectorizer,
    KnowledgeSourceIngestionParameters,
)

knowledge_source = AzureBlobKnowledgeSource(
    name=ks_name,
    azure_blob_parameters=AzureBlobKnowledgeSourceParameters(
        connection_string=managed_identity_connection,
        container_name=source_container,
        ingestion_parameters=KnowledgeSourceIngestionParameters(
            content_extraction_mode="standard",
            embedding_model=KnowledgeSourceAzureOpenAIVectorizer(
                azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
                    resource_url=foundry_endpoint,
                    deployment_name=foundry_embedding_deployment,
                    model_name=foundry_embedding_model,
                )
            ),
            chat_completion_model=KnowledgeBaseAzureOpenAIModel(
                azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
                    resource_url=foundry_endpoint,
                    deployment_name=foundry_chat_deployment,
                    model_name=foundry_chat_model,
                )
            ),
            disable_image_verbalization=False,
            ai_services=AIServices(uri=foundry_endpoint),
            asset_store=AssetStore(
                connection_string=managed_identity_connection,
                container_name=asset_container,
            ),
        ),
    ),
)

result = index_client.create_or_update_knowledge_source(
    knowledge_source=knowledge_source
)
if not isinstance(result, AzureBlobKnowledgeSource):
    raise TypeError("Expected an Azure Blob knowledge source response.")
created_resources = result.azure_blob_parameters.created_resources
if created_resources is None:
    raise RuntimeError(
        "The knowledge source response didn't include generated resources."
    )
generated_index_name = created_resources["index"]
print(f"Knowledge source '{result.name}' created or updated.")
print(f"Generated index: {generated_index_name}")

## Wait for managed ingestion

The Azure Blob knowledge source synchronizes asynchronously. Poll its status until the first synchronization completes. Treat failed items as an error instead of continuing with an incomplete index.


In [ ]:
POLL_INTERVAL_SECONDS = 30
POLL_TIMEOUT_SECONDS = 1800

deadline = time.monotonic() + POLL_TIMEOUT_SECONDS
while True:
    status = index_client.get_knowledge_source_status(ks_name)
    current = status.current_synchronization_state
    if current is not None:
        print(
            f"Managed ingestion {status.synchronization_status}: "
            f"{current.items_updates_processed} processed, "
            f"{current.items_updates_failed} failed, "
            f"{current.items_skipped} skipped."
        )
        if current.items_updates_failed:
            messages = [error.error_message for error in current.errors or []]
            raise RuntimeError(
                "Managed ingestion has failed items.\n" + "\n".join(messages)
            )
    completed = status.last_synchronization_state
    if current is None and completed is not None:
        if completed.items_updates_failed:
            raise RuntimeError(
                "Managed ingestion completed with "
                f"{completed.items_updates_failed} failed item(s)."
            )
        print(
            "Managed ingestion completed: "
            f"{completed.items_updates_processed} item(s) processed."
        )
        break
    if time.monotonic() >= deadline:
        raise TimeoutError("Timed out waiting for managed ingestion.")
    time.sleep(POLL_INTERVAL_SECONDS)

## Verify the generated index

Managed ingestion creates the index schema and populates `image_path` for chunks associated with extracted images. Independently query the generated index and stop if no indexed image path is available.


In [ ]:
from azure.search.documents import SearchClient

search_client = SearchClient(
    endpoint=endpoint,
    index_name=generated_index_name,
    credential=credential,
)
documents = list(
    search_client.search(
        search_text="*",
        select=["blob_url", "snippet", "image_path"],
        top=100,
    )
)
selected_image_path = None
for document in documents:
    paths = document.get("image_path") or []
    if paths:
        selected_image_path = paths[0] if isinstance(paths, list) else paths
        break

if not selected_image_path:
    raise RuntimeError(
        "The generated index doesn't contain a nonempty image_path. "
        "Verify that ingestion succeeded and the source contains extractable images."
    )

print(f"Selected indexed image_path: {selected_image_path}")

## Create a knowledge base with image serving enabled

Set `enable_image_serving=True` on the blob knowledge-source reference. Azure AI Search uses matching asset-store images as multimodal input during answer synthesis, but the retrieve response doesn't return the image bytes.


In [ ]:
from azure.search.documents.indexes.models import (
    KnowledgeBase,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalMediumReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

knowledge_base = KnowledgeBase(
    name=kb_name,
    knowledge_sources=[
        KnowledgeSourceReference(
            name=ks_name,
            enable_image_serving=True,
        )
    ],
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    retrieval_reasoning_effort=KnowledgeRetrievalMediumReasoningEffort(),
    models=[
        KnowledgeBaseAzureOpenAIModel(
            azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
                resource_url=foundry_endpoint,
                deployment_name=foundry_chat_deployment,
                model_name=foundry_chat_model,
            )
        )
    ],
)

index_client.create_or_update_knowledge_base(knowledge_base=knowledge_base)
print(f"Knowledge base '{kb_name}' created with image serving enabled.")

In [ ]:
from azure.search.documents.knowledgebases.models import (
    AzureBlobKnowledgeSourceParams,
    KnowledgeBaseAzureBlobActivityRecord,
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
)

query = os.getenv(
    "AZURE_SEARCH_QUERY",
    "What information is shown in the diagrams and images?",
)


def retrieve(enable_image_serving):
    request = KnowledgeBaseRetrievalRequest(
        messages=[
            KnowledgeBaseMessage(
                role="user",
                content=[KnowledgeBaseMessageTextContent(text=query)],
            )
        ],
        knowledge_source_params=[
            AzureBlobKnowledgeSourceParams(
                knowledge_source_name=ks_name,
                enable_image_serving=enable_image_serving,
            )
        ],
        include_activity=True,
        output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    )
    return kb_retrieval_client.retrieve(request)


def answer_text(retrieval_response):
    if not retrieval_response.response:
        return ""
    return "\n".join(
        content.text or ""
        for message in retrieval_response.response
        for content in message.content or []
        if isinstance(content, KnowledgeBaseMessageTextContent)
    )


def get_image_serving_totals(retrieval_response):
    statistics = [
        record.image_serving
        for record in retrieval_response.activity or []
        if isinstance(record, KnowledgeBaseAzureBlobActivityRecord)
        and record.image_serving is not None
    ]
    return {
        "images_retrieved": sum(item.images_retrieved or 0 for item in statistics),
        "images_sent_to_model": sum(
            item.images_sent_to_model or 0 for item in statistics
        ),
        "total_image_size_bytes": sum(
            item.total_image_size_bytes or 0 for item in statistics
        ),
        "verbalization_used": any(
            item.verbalization_used is True for item in statistics
        ),
    }


disabled_response = retrieve(enable_image_serving=False)
for attempt in range(1, 6):
    response = retrieve(enable_image_serving=True)
    if get_image_serving_totals(response)["images_sent_to_model"] > 0:
        break
    print(f"Image serving isn't ready (attempt {attempt} of 5). Retrying.")
    time.sleep(POLL_INTERVAL_SECONDS)

disabled_answer = answer_text(disabled_response)
enabled_answer = answer_text(response)
print("Without image serving:")
print(disabled_answer)
print("\nWith image serving:")
print(enabled_answer)

## Inspect image serving statistics

Each `KnowledgeBaseAzureBlobActivityRecord` in the response carries an `image_serving` field of type
`ImageServingStatistics` when image serving is active. The statistics show how many images were
retrieved from the asset store, how many were passed to the model, and the total image size.
The `verbalization_used` value reflects the knowledge source's indexing-time image verbalization
configuration and state; it isn't a retrieval fallback.


In [ ]:
disabled_totals = get_image_serving_totals(disabled_response)
enabled_totals = get_image_serving_totals(response)
print(f"Image serving disabled: {disabled_totals}")
print(f"Image serving enabled:  {enabled_totals}")

assert disabled_totals["images_sent_to_model"] == 0
assert enabled_totals["images_retrieved"] > 0
assert enabled_totals["images_sent_to_model"] > 0
assert enabled_totals["total_image_size_bytes"] > 0

## Download an indexed image asset

The retrieve response reports aggregate image-serving activity but doesn't define dedicated fields for individual asset-store image paths or image bytes sent to the model. The earlier wildcard index query selected `selected_image_path` independently of retrieval. Use the application identity, which needs **Storage Blob Data Reader**, to download that indexed asset. This cell verifies that the blob is nonempty and has an image content type.


In [ ]:
from urllib.parse import unquote, urlparse

from azure.storage.blob import BlobServiceClient

storage_account_url = os.environ["AZURE_STORAGE_ACCOUNT_URL"]
indexed_image_path = str(selected_image_path).split(";", maxsplit=1)[0]
parsed_path = urlparse(indexed_image_path)
if parsed_path.scheme and parsed_path.netloc:
    decoded_path = unquote(parsed_path.path).lstrip("/")
    container_prefix = f"{asset_container}/"
    blob_name = (
        decoded_path[len(container_prefix) :]
        if decoded_path.lower().startswith(container_prefix.lower())
        else decoded_path
    )
else:
    blob_name = indexed_image_path
    if ":" in blob_name:
        blob_name = blob_name.split(":", maxsplit=1)[1]
    blob_name = blob_name.lstrip("/")

blob_service_client = BlobServiceClient(
    account_url=storage_account_url,
    credential=credential,
)
blob_client = blob_service_client.get_blob_client(
    container=asset_container,
    blob=blob_name,
)
properties = blob_client.get_blob_properties()
image_bytes = blob_client.download_blob().readall()
content_type = properties.content_settings.content_type or ""

assert image_bytes, "The selected indexed image blob is empty."
assert content_type.startswith(
    "image/"
), f"Expected an image content type, but received '{content_type}'."
print(f"Downloaded {len(image_bytes)} bytes with content type {content_type}.")

## Clean up

Delete the knowledge base first, then the knowledge source. Skip these cells when you want to retain the generated Search pipeline. Deleting the Search resources doesn't delete source documents or projected image blobs. Delete those blobs separately only when no retained ingestion or retrieval pipeline still needs them.


### Delete knowledge base


In [ ]:
from azure.core.exceptions import ResourceNotFoundError

try:
    index_client.delete_knowledge_base(kb_name)
    print(f"Knowledge base '{kb_name}' deleted.")
except ResourceNotFoundError:
    print(f"Knowledge base '{kb_name}' doesn't exist; nothing to delete.")

### Delete knowledge source


In [ ]:
try:
    index_client.delete_knowledge_source(knowledge_source=ks_name)
    print(f"Knowledge source '{ks_name}' deleted.")
except ResourceNotFoundError:
    print(f"Knowledge source '{ks_name}' doesn't exist; nothing to delete.")